# Week 3 — Data completeness / gap report
Checks every downloaded station's actual data coverage against the expected
1 Aug - 30 Sep 2019 window, counts real gaps, and flags any MiniSEED
integrity warnings encountered while reading (e.g. the Steim2 warning seen
on NTVZ) so we know whether they affect usable data volume or not.

In [ ]:
import os, glob, warnings
import obspy
from obspy import read, UTCDateTime
from obspy.io.mseed.headers import InternalMSEEDWarning
import pandas as pd

PROJECT_DIR = os.path.expanduser("~/tvz_project")
WAVEFORM_DIR = os.path.join(PROJECT_DIR, "data", "waveforms")

STARTTIME = UTCDateTime("2019-08-01T00:00:00")
ENDTIME = UTCDateTime("2019-09-30T23:59:59")
EXPECTED_SECONDS = ENDTIME - STARTTIME

station_dirs = sorted(glob.glob(os.path.join(WAVEFORM_DIR, "NZ.*")))
print(f"{len(station_dirs)} station directories found")

In [ ]:
rows = []

for station_dir in station_dirs:
    sta_code = os.path.basename(station_dir).split(".")[1]
    mseed_files = sorted(glob.glob(os.path.join(station_dir, "*.mseed")))

    st = obspy.Stream()
    n_warnings = 0
    for f in mseed_files:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            st += read(f)
            n_warnings += sum(
                1 for w in caught if issubclass(w.category, InternalMSEEDWarning)
            )

    st.merge(method=0)  # combines contiguous segments; genuine gaps stay as separate traces

    channel_ids = sorted({tr.id for tr in st})
    for chan_id in channel_ids:
        chan_traces = [tr for tr in st if tr.id == chan_id]
        covered_seconds = sum(tr.stats.endtime - tr.stats.starttime for tr in chan_traces)
        n_gaps = max(len(chan_traces) - 1, 0)
        completeness = 100 * covered_seconds / EXPECTED_SECONDS
        rows.append({
            "station": sta_code,
            "channel": chan_id,
            "n_files": len(mseed_files),
            "covered_days": round(covered_seconds / 86400, 2),
            "completeness_pct": round(completeness, 1),
            "n_gaps": n_gaps,
            "n_mseed_warnings": n_warnings,
        })

df = pd.DataFrame(rows).sort_values("completeness_pct")
print(f"{len(df)} channel-records checked across {len(station_dirs)} stations")

In [ ]:
# Full table, worst completeness first
pd.set_option("display.max_rows", None)
print(df.to_string(index=False))

out_csv = os.path.join(PROJECT_DIR, "data", "gap_report_2019-08-01_2019-09-30.csv")
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
# Summary + specific check on NTVZ (the station that threw Steim2 warnings)
print("=== Summary ===")
print(f"Mean completeness: {df['completeness_pct'].mean():.1f}%")
print(f"Channels below 95% complete: {(df['completeness_pct'] < 95).sum()} of {len(df)}")
print(f"Channels with MiniSEED integrity warnings: {(df['n_mseed_warnings'] > 0).sum()} of {len(df)}")

print("\n=== NTVZ specifically (the Steim2 warning from the download log) ===")
print(df[df["station"] == "NTVZ"].to_string(index=False))